In [1]:
!pip install -U transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 91.0 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.8/558.8 kB 38.3 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.33.1
    Uninstalling huggingface-hub-0.33.1:
      Successfully uninstalled huggingface-hub-0.33.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.52.4
    Uninstalling transformers-4.52.4:
      Successfully uninstalled transformers-4.52.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 3.6.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2025.5.1 which is incompatible.


In [3]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("GITHUB_TOKEN")
repo_url = "github.com/tanishka4481/EgoLoc"
!git clone https://$token@$repo_url

Cloning into 'EgoLoc'...
remote: Enumerating objects: 275, done.
remote: Total 275 (delta 0), reused 0 (delta 0), pack-reused 275 (from 1)
Receiving objects: 100% (275/275), 236.99 MiB | 42.33 MiB/s, done.
Resolving deltas: 100% (130/130), done.
Updating files: 100% (47/47), done.


In [4]:
import cv2
import numpy as np
import os
import math
from PIL import Image
from transformers import AutoProcessor, AutoModelForVision2Seq
import torch
import gc

def image_resize(image, width=None, height=None, inter=cv2.INTER_AREA):
    """Resize *image* while keeping aspect ratio.

    Args:
        image (np.ndarray): BGR image to resize.
        width (int | None): Desired width. If None, compute from *height*.
        height (int | None): Desired height. If None, compute from *width*.
        inter: OpenCV interpolation method.

    Returns:
        np.ndarray: Resized image.
    """
    dim = None
    (h, w) = image.shape[:2]
    if width is None and height is None:
        return image
    if width is None:
        r = height / float(h)
        dim = (int(w * r), height)
    else:
        r = width / float(w)
        dim = (width, int(h * r))
    resized = cv2.resize(image, dim, interpolation=inter)
    return resized

def create_frame_grid_from_indices(video_path, frame_indices, grid_size):
    """
    Build a grid image from *frame_indices* and draw numbered circles.

    Args:
        video_path (str): Video file.
        frame_indices (list[int]): List of frames to show.
        grid_size (int): Grid side length.

    Returns:
        np.ndarray: Grid image (uint8 BGR).
    """
    spacer = 0
    video = cv2.VideoCapture(video_path)
    if not video.isOpened():
        raise IOError(f"Could not open video file: {video_path}")

    frames_for_this_grid = grid_size**2
    indices_to_process = frame_indices[:frames_for_this_grid]

    frames = []
    for idx, index in enumerate(indices_to_process):
        video.set(cv2.CAP_PROP_POS_FRAMES, index)
        success, frame = video.read()
        if success:
            frame = image_resize(frame, width=200)
            frames.append(frame)
        else:
            if frames:
                black_frame = np.zeros_like(frames[0])
            else:
                black_frame = np.zeros((200, 200, 3), dtype=np.uint8)
            frames.append(black_frame)
            print(f"Warning: Could not read frame {index}. Appending a black placeholder for grid cell {idx+1}.")
    video.release()

    if len(frames) < frames_for_this_grid:
        print(f"Padding grid with {frames_for_this_grid - len(frames)} black frames.")
        if frames:
            black_frame_template = np.zeros_like(frames[0])
        else:
            black_frame_template = np.zeros((200, 200, 3), dtype=np.uint8)
        frames.extend([black_frame_template] * (frames_for_this_grid - len(frames)))

    if not frames:
        raise ValueError("No frames could be read or created to form the grid.")
    
    target_height, target_width = frames[0].shape[:2]
    for i in range(len(frames)):
        if frames[i].shape[:2] != (target_height, target_width):
            frames[i] = cv2.resize(frames[i], (target_width, target_height))

    frame_height, frame_width = frames[0].shape[:2]
    grid_height = grid_size * frame_height + (grid_size - 1) * spacer
    grid_width = grid_size * frame_width + (grid_size - 1) * spacer
    grid_img = np.ones((grid_height, grid_width, 3), dtype=np.uint8) * 255

    for i in range(grid_size):
        for j in range(grid_size):
            current_grid_cell_idx = i * grid_size + j
            frame = frames[current_grid_cell_idx]

            if current_grid_cell_idx < len(indices_to_process):
                actual_frame_number = indices_to_process[current_grid_cell_idx]
                max_dim = int(min(frame.shape[:2]) * 0.5)
                overlay = frame.copy()
                circle_center = (frame.shape[1] - max_dim // 2, max_dim // 2)
                cv2.circle(overlay, circle_center, max_dim // 2, (255, 255, 255), -1)
                alpha = 0.3
                frame = cv2.addWeighted(overlay, alpha, frame, 1 - alpha, 0)
                cv2.circle(frame, circle_center, max_dim // 2, (255, 255, 255), 2)
                font_scale = max_dim / 50
                text_to_display = str(actual_frame_number)
                text_size = cv2.getTextSize(
                    text_to_display, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 2)[0]
                text_x = frame.shape[1] - text_size[0] // 2 - max_dim // 2
                text_y = text_size[1] // 2 + max_dim // 2
                cv2.putText(frame, text_to_display, (text_x, text_y),
                            cv2.FONT_HERSHEY_SIMPLEX, font_scale, (0, 0, 0), 2)

            y1 = i * (frame_height + spacer)
            y2 = y1 + frame_height
            x1 = j * (frame_width + spacer)
            x2 = x1 + frame_width
            grid_img[y1:y2, x1:x2] = frame
    return grid_img

def read_clip_ranges(file_path):
    """
    Reads frame ranges from a text file.
    Each line is expected to be in the format: '1. 79, 120'
    Returns a list of tuples, e.g., [(79, 120), ...]
    """
    clip_ranges = []
    with open(file_path, 'r') as f:
        for line in f:
            try:
                parts = line.strip().split('.')
                if len(parts) > 1:
                    frame_str = parts[1].strip()
                    if ',' not in frame_str:
                        start_frame = int(frame_str)
                        clip_ranges.append((start_frame, start_frame))
                    else:
                        start_frame, end_frame = map(int, frame_str.split(','))
                        clip_ranges.append((start_frame, end_frame))
            except ValueError as e:
                print(f"Warning: Could not parse line '{line.strip()}'. Skipping. Error: {e}")
    return clip_ranges

def split_ranges_to_grids(ranges, max_frames_per_grid=16):
    """
    Splits a list of (start, end) frame ranges into smaller sub-ranges
    that fit within a single grid (max_frames_per_grid).
    """
    all_grids = []
    for start, end in ranges:
        grids = []
        current_start = start
        while current_start <= end:
            current_end = min(current_start + max_frames_per_grid - 1, end)
            grids.append((current_start, current_end))
            current_start = current_end + 1
        all_grids.append(grids)
    return all_grids

def grid_prompt(start_frame, end_frame):
    """Generates a prompt for the VLM based on the grid's frame range."""
    prompt = (
        f"You are shown a numbered, consecutive sequence of video frames ({start_frame} to {end_frame}) arranged in a grid. A human hand may interact with one or more objects.""""
    Your task: For this grid, perform the following:
    Identify the main object:
    Carefully examine all frames and describe (in a single short sentence) the most purposefully manipulated object by the hand—using color, shape, size, or shelf location (e.g., “blue bottle on left shelf”).
    1.Frame-by-frame annotation:
    Step through the frames in order.
    For each frame, note the hand’s position and action related to the main object: (a) Approaching, (b) First clear physical contact/grasp/touch, (c) Holding or manipulating, (d) Releasing/moving away.
    
    2.Boundary handling:
    If the hand is already holding/touching the object in the first frame, do NOT mark this as contact unless a new grasp starts here. Contact likely began before this grid.
    If the hand is still holding/touching the object in the last frame, do NOT mark separation unless you see a clear release or movement away. Separation may happen after this grid.
    If contact or separation is ambiguous or occurs at the grid’s edge, clearly state so with an explanation.
    3.Padding/blank frame handling:
    If the grid contains mostly black or blank frames, or you are unable to see relevant actions, respond: “Grid contains mostly padding; no meaningful hand-object interaction detected.”
    4.Uncertainty:
    If you cannot decisively identify a main object or hand action, say: “Uncertain which object is the main focus in this grid.”

    When determining separation, look for the first frame where the hand has just begun to leave the object—even if only a slight gap appears. Do NOT wait until the hand is far away; 
    mark separation at the earliest visible sign that fingers or palm are no longer in full contact with the object, even if the hand is still very close by. If separation is ambiguous, 
    err on the side of marking the earliest possible frame of disengagement.

    CRITICAL: Only declare 'contact' when the hand or fingertips actually touch the object—not merely when the hand appears in the same line as the object. 
    Look for visual cues such as the finger or hand edge physically overlapping the object, changes in finger shape, or shadows/occlusion indicating touch. 
    If you cannot clearly see touch, especially when the hand is approaching from a distance, do not mark contact yet. Sometimes, the user only holds the object with their fingers, 
    not the entire hand—mark 'contact' as soon as any one finger touches, even if the palm is not involved
    
    Output Format:
    {
      "object_description": "...",
      "contact_frame": ...,           // Frame number, or null if no new contact in this grid
      "separation_frame": ...,        // Frame number, or null if no release in this grid
      "reasoning": "Step-by-step rationale, referencing specific frames and transitions. Explain if/why contact/separation is not in this grid or is ambiguous."
    }
    Example (do NOT copy numbers):
    "The hand approaches the blue box at frame N, touches it at frame M, holds it from frame M to P, and lets go at frame Q, as the hand moves away from the object. If the release is not visible here, state 'hand still holding at last frame; release occurs later."""
    )
    
    return prompt

def combine_clip_answers(clip_grid_answers):
    """Combines VLM answers from multiple grids for a single clip."""
    combined_summary = ""
    for ans in clip_grid_answers:
        combined_summary += (
            f"Grid {ans['grid_range'][0]}-{ans['grid_range'][1]}:\n"
            f"{ans['answer']}\n\n"
        )
    return combined_summary.strip()

def run_vlm_on_grid(grid_img, prompt, model, processor):
    """
    Performs Qwen VLM inference on a single grid image object.
    
    Args:
        grid_img (PIL.Image.Image): The grid image object.
        prompt (str): The prompt for the VLM.
        model: The loaded VLM model.
        processor: The VLM processor.
    """
    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": grid_img},
                    {"type": "text", "text": prompt}
                ]
            },
        ]

        # Apply chat template and tokenize
        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(**inputs, max_new_tokens=200)

        answer = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        
        return answer, outputs, inputs
    except Exception as e:
        return f"VLM inference failed on grid: {e}", None, None

def main():
    video_path = "/kaggle/working/EgoLoc/test_data/first10clips.mp4"
    clip_ranges_file = "/kaggle/working/EgoLoc/test_data/clip_ranges_10.txt"
    output_dir = "/kaggle/working/output_grids" 
    fixed_grid_size = 4
    
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    
    # Use llava-hf/llava-1.5-7b-hf or Qwen/Qwen2.5-VL-3B-Instruct
    
    processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
    model = AutoModelForVision2Seq.from_pretrained(
        "Qwen/Qwen2.5-VL-7B-Instruct",    
        device_map="auto",
        torch_dtype=torch.float16 if device == "cuda" else torch.float32
    )
    print("Model loaded.")

    os.makedirs(output_dir, exist_ok=True)

    clip_intervals_from_file = read_clip_ranges(clip_ranges_file)
    if not clip_intervals_from_file:
        print(f"No clip ranges found in '{clip_ranges_file}'. Exiting.")
        return

    frames_per_grid = fixed_grid_size * fixed_grid_size # 16 frames per grid
    all_clip_grids_ranges = split_ranges_to_grids(clip_intervals_from_file, frames_per_grid)

    all_results = []
    
    for clip_idx, grids_for_clip in enumerate(all_clip_grids_ranges):
        clip_result = []
        start_frame_of_clip, end_frame_of_clip = clip_intervals_from_file[clip_idx]
        
        print(f"\n--- Processing Clip {clip_idx} (Frames {start_frame_of_clip}-{end_frame_of_clip}) ---")
        
        for grid_num, (start, end) in enumerate(grids_for_clip):
            current_grid_frame_indices = list(range(start, end + 1))
            
            print(f"Creating Grid {grid_num + 1} for frames: {current_grid_frame_indices}")
            
            try:
                grid_image_bgr = create_frame_grid_from_indices(video_path, current_grid_frame_indices, fixed_grid_size)
                
                grid_image_pil = Image.fromarray(cv2.cvtColor(grid_image_bgr, cv2.COLOR_BGR2RGB))
                
                video_name = os.path.splitext(os.path.basename(video_path))[0]
                output_image_name = f"{video_name}_clip_{clip_idx}_grid_{start}-{end}_part{grid_num+1}.jpg"
                output_image_path = os.path.join(output_dir, output_image_name)
                
                grid_image_pil.save(output_image_path)
                print(f"Grid image saved to: {output_image_path}")

                prompt = grid_prompt(start, end)
                # Pass the PIL Image object directly and the prompt
                answer, outputs, inputs = run_vlm_on_grid(grid_image_pil, prompt, model, processor)
                print(f"VLM Output for grid {start}-{end}:\n{answer}\n")
                
                clip_result.append({"grid_range": (start, end), "answer": answer})
                
            except Exception as e:
                print(f"An error occurred while processing grid {grid_num + 1} for Clip {clip_idx}: {e}")
                answer = f"An error occurred: {e}"
                clip_result.append({"grid_range": (start, end), "answer": answer})
            
            if 'outputs' in locals() and outputs is not None:
                del outputs
            if 'inputs' in locals() and inputs is not None:
                del inputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
            print("Memory cleaned up.")
                
        all_results.append(clip_result)
    
    print("\n--- Final Combined Summaries ---")
    all_clip_summaries = [combine_clip_answers(g) for g in all_results]
    for clip_idx, summary in enumerate(all_clip_summaries):
        print(f"=== Clip {clip_idx} Combined Summary ===\n{summary}\n")
    del model
    del processor
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return all_clip_summaries

if __name__ == "__main__":
    results = main()

2025-08-02 07:58:41.830053: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754121522.128899      36 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754121522.210146      36 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Using device: cuda
Loading Qwen model...


preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model-00005-of-00005.safetensors:   0%|          | 0.00/1.09G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

Model loaded.

--- Processing Clip 0 (Frames 79-120) ---
Creating Grid 1 for frames: [79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94]
Grid image saved to: /kaggle/working/output_grids/first10clips_clip_0_grid_79-94_part1.jpg
VLM Output for grid 79-94:
{
  "object_description": "blue bottle on left shelf",
  "contact_frame": 80,
  "separation_frame": 94,
  "reasoning": "The hand approaches the blue bottle on the left shelf in frame 79. It makes contact with the bottle in frame 80, holding it from frame 80 to 93. The hand releases the bottle in frame 94, as indicated by the hand moving away without any further contact."
}

Memory cleaned up.
Creating Grid 2 for frames: [95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110]
Grid image saved to: /kaggle/working/output_grids/first10clips_clip_0_grid_95-110_part2.jpg
VLM Output for grid 95-110:
{
  "object_description": "Blue object on the top shelf",
  "contact_frame": 96,
  "separation_frame": 108,
  "r

In [5]:
with open("/kaggle/working/output_qwen_final.txt", "a+") as f:
    for i in results:
        f.write(i)